In [1]:
##### Cleans India capital stock data
# reformats

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds
from rasterio.warp import reproject, Resampling
import rasterstats
from rasterstats import zonal_stats

In [2]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())

# Import data
capital_stock = pd.read_csv(f"{cd}/Data/Raw/Sub_National/India/tractors_2011.csv")

IND_codes = pd.read_csv(f"{cd}/Data/Correspondence_tables/IND_states.csv")

IND_states = gpd.read_file('/Users/carinamanitius/Documents/Data/Admin_Boundaries/India/India_states.shp')

ag_production = f"{cd}/Data/Clean/Production/total_production_USD_2020.tif"

# Set save path
save_path = f"{cd}/Data/Clean/Capital_stock/IND_capital_stock.csv"

In [3]:
##### Clean

# retitle column 
capital_stock = capital_stock.rename(columns={'number_of_tractors': '2011'})

# add units 
capital_stock['Units'] = 'Ag capital stock - count of tractors'

# reorder columns
columns_to_keep = ['GID_1', 'NAME_1', 'Units', '2011']
capital_stock = capital_stock[columns_to_keep]

In [4]:
### Final cleaning 

capital_stock['ISO3'] = 'IND'
capital_stock['Year'] = 2011
capital_stock['GEO_ID_NAME'] = 'GID_1'

capital_stock = capital_stock.rename(columns={'2011': 'ag_capital_stock_count_machinery', 'GID_1': 'GEO_ID'})

col_to_keep = ['ISO3', 'GEO_ID', 'GEO_ID_NAME', 'Year', 'ag_capital_stock_count_machinery']
capital_stock = capital_stock[col_to_keep]

In [5]:
##### Fill missing state data 

### 1: identify missing states 
IND_states = IND_states.merge(capital_stock, left_on='GID_1', right_on='GEO_ID', how='left')

IND_states_missing = IND_states[IND_states['ag_capital_stock_count_machinery'].isna()].copy()
IND_states_filled  = IND_states[IND_states['ag_capital_stock_count_machinery'].notna()].copy()

### 2: calculate the total ag production in each state
def subnational_production_stats(geo, raster_path, column_name, nodata=-9999):

    with rasterio.open(raster_path) as src:
        raster_crs = src.crs

    # ensure crs match
    gdf_proj = geo.to_crs(raster_crs)

    zonal = rasterstats.zonal_stats(
        gdf_proj,
        raster_path,
        stats="sum",
        nodata=nodata
    )

    geo[column_name] = [z["sum"] for z in zonal]
    return geo

IND_states_missing_prod = subnational_production_stats(IND_states_missing, ag_production, "total_production_USD")
IND_states_filled_prod = subnational_production_stats(IND_states_filled, ag_production, "total_production_USD")



In [ ]:
IND_states_filled_prod

In [6]:
##### Save cleaned data
capital_stock.to_csv(save_path, index=False)